# Model Building — Phishing Website Detector

**Projeto:** Deteção de Websites de Phishing através de Machine Learning  
**Fase:** Model Building (CRISP-ML(Q))  
**Curso:** Inteligência Artificial Aplicada à Cibersegurança  
**Data:** Setembro de 2026

Nesta fase são desenvolvidos e comparados quatro modelos de classificação supervisionada:
1. Logistic Regression (baseline)
2. Decision Tree
3. Random Forest
4. XGBoost (obrigatório)

**Referência:** [GeeksforGeeks Machine Learning Tutorial](https://www.geeksforgeeks.org/machine-learning/machine-learning-tutorial/) — Module 6 (Supervised Learning, Ensemble Learning, XGBoost) e Module 7 (Confusion Matrix, Precision, Recall, F1-Score, AUC-ROC, Cross-validation).

## Preparação do ambiente

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

PASTA_PROJETO = Path.cwd().parent
sys.path.insert(0, str(PASTA_PROJETO / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from model_utils import (
    carregar_dataset_preparado,
    separar_features_target,
    dividir_treino_teste,
    criar_logistic_regression,
    criar_decision_tree,
    criar_random_forest,
    criar_xgboost,
    treinar_modelo,
    prever,
    prever_probabilidades,
    avaliar_modelo,
    cross_validation,
    comparar_treino_teste,
    comparar_modelos,
    importancia_features,
    grafico_matriz_confusao,
    grafico_comparacao,
    grafico_curvas_roc,
    grafico_curvas_pr,
    grafico_feature_importance,
)

print("Bibliotecas carregadas com sucesso.")

## 1. Importação do dataset preparado

In [ ]:
df = carregar_dataset_preparado("phishing_prepared.csv")
df.head()

## 2. Separação de features e target

In [ ]:
X, y = separar_features_target(df, "class")

# Mapear classes: -1 -> 0 (phishing), 1 -> 1 (legítimo)
y = y.map({-1: 0, 1: 1})

print("Distribuição das classes:")
print(y.value_counts())

## 3. Divisão treino e teste

In [ ]:
X_train, X_test, y_train, y_test = dividir_treino_teste(X, y)
nomes_features = X.columns.tolist()

## 4. Criação dos modelos

In [ ]:
logistic_regression = criar_logistic_regression()
decision_tree = criar_decision_tree()
random_forest = criar_random_forest()
xgboost = criar_xgboost()

modelos = [
    ("Logistic Regression", logistic_regression),
    ("Decision Tree", decision_tree),
    ("Random Forest", random_forest),
    ("XGBoost", xgboost),
]

for nome, modelo in modelos:
    print(f"  • {nome}")

## 5. Treino dos modelos

In [ ]:
for nome, modelo in modelos:
    treinar_modelo(modelo, X_train, y_train)
    print(f"  • {nome} treinado.")

## 6. Previsões e avaliação no conjunto de teste

In [ ]:
resultados = []

for nome, modelo in modelos:
    y_pred = prever(modelo, X_test)
    y_pred_proba = prever_probabilidades(modelo, X_test)

    resultado, matriz, relatorio = avaliar_modelo(
        nome, y_test, y_pred, y_pred_proba,
        nomes_classes=["Phishing (0)", "Legítimo (1)"]
    )

    resultado["y_pred"] = y_pred
    resultado["y_pred_proba"] = y_pred_proba
    resultado["matriz"] = matriz
    resultados.append(resultado)

## 7. Cross-validation (5-fold)

In [ ]:
for nome, modelo in modelos:
    print(f"
{nome}:")
    cv_resultados = cross_validation(modelo, X_train, y_train, cv_folds=5)

## 8. Comparação treino vs teste (overfitting)

In [ ]:
for nome, modelo in modelos:
    print(f"
{nome}:")
    comparar_treino_teste(modelo, X_train, y_train, X_test, y_test)

## 9. Comparação dos modelos

In [ ]:
resultados_tabela = []
for r in resultados:
    linha = {k: v for k, v in r.items() if k not in ["y_pred", "y_pred_proba", "matriz"]}
    resultados_tabela.append(linha)

tabela = comparar_modelos(resultados_tabela)
tabela

## 10. Importância das features

In [ ]:
for nome, modelo in modelos:
    importancias = importancia_features(modelo, nomes_features)
    if importancias is not None:
        print(f"
--- TOP 10 — {nome} ---")
        print(importancias.head(10))
        
        fig = grafico_feature_importance(importancias, nome, top_n=15)
        plt.show()

## 11. Matrizes de confusão

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, resultado in zip(axes.flat, resultados):
    grafico_matriz_confusao(resultado["matriz"], resultado["Modelo"], ax=ax)
plt.suptitle("Matrizes de Confusão — Comparação de Modelos", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96], h_pad=3.0)
plt.show()

## 12. Comparação de métricas

In [ ]:
for metrica in ["Accuracy", "Precision", "Recall", "F1-score"]:
    fig = grafico_comparacao(resultados_tabela, metrica=metrica)
    plt.show()

## 13. Curvas ROC

In [ ]:
fig = grafico_curvas_roc(resultados, y_test)
plt.show()

## 14. Curvas Precision-Recall

In [ ]:
fig = grafico_curvas_pr(resultados, y_test)
plt.show()

## 15. Conclusão

O modelo **XGBoost** demonstrou o melhor desempenho global, com Accuracy de 96,94%, Recall de 98,01% e apenas 11 falsos negativos no conjunto de teste.

As features mais importantes para a deteção de phishing foram **HTTPS**, **AnchorURL** e **PrefixSuffix-**, consistentes em todos os modelos baseados em árvores.

A comparação treino vs teste não revelou overfitting significativo em nenhum dos modelos.

Os próximos passos incluem otimização de hiperparâmetros e avaliação do impacto da remoção de features altamente correlacionadas.